# First Best Model

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import recall_score, precision_score, classification_report

pd.set_option("display.max_columns", None)

# 1. LOAD DATA
train_df = pd.read_csv("/kaggle/input/anava-ugm/train.csv")
test_df  = pd.read_csv("/kaggle/input/anava-ugm/test.csv")

TARGET = "Trip_Label"
ID_COL = "Trip_ID"

# 2. FEATURE ENGINEERING 
full = pd.concat(
    [train_df.drop(columns=[TARGET]), test_df],
    axis=0
).reset_index(drop=True)

full["Timestamp"] = pd.to_datetime(full["Timestamp"], errors="coerce")
full["Timestamp"] = full["Timestamp"].fillna(full["Timestamp"].median())

for c in ["Pickup_Lat","Pickup_Long","Dropoff_Lat","Dropoff_Long"]:
    full[c] = full[c].fillna(full[c].median())

full["Battery_Level"] = (
    full["Battery_Level"].astype(str)
    .str.replace("%","", regex=False)
)
full["Battery_Level"] = pd.to_numeric(full["Battery_Level"], errors="coerce")
full["Battery_Level"] = full["Battery_Level"].fillna(full["Battery_Level"].median())

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2-lon1, lat2-lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

full["haversine_km"] = haversine(
    full["Pickup_Long"], full["Pickup_Lat"],
    full["Dropoff_Long"], full["Dropoff_Lat"]
)

full["dist_ratio"] = full["haversine_km"] / (full["Distance_KM"] + 1e-3)

full["Surge_Multiplier"] = full["Surge_Multiplier"].fillna(1.0)
full["price_per_km"] = full["Est_Price_IDR"] / (full["Distance_KM"] + 1e-3)
full["surge_flag"] = (full["Surge_Multiplier"] > 1).astype(int)

zone_avg = full.groupby("Pickup_Zone")["price_per_km"].transform("mean")
full["price_vs_zone"] = full["price_per_km"] / (zone_avg + 1e-3)

full["promo_flag"] = full["Promo_Code"].notna().astype(int)
full.drop(columns=["Promo_Code"], inplace=True)

full["Hour"] = full["Timestamp"].dt.hour
full["DayOfWeek"] = full["Timestamp"].dt.dayofweek
full["IsWeekend"] = (full["DayOfWeek"] >= 5).astype(int)
full.drop(columns=["Timestamp"], inplace=True)

full["is_normal_dist"]  = (np.abs(full["dist_ratio"] - 1) < 0.15).astype(int)
full["is_normal_price"] = (np.abs(full["price_vs_zone"] - 1) < 0.15).astype(int)

cat_cols = [
    "Pickup_Zone","Dropoff_Zone","Device_FP","Car_Model",
    "Payment_Method","Weather","Traffic","Signal_Strength"
]
for c in cat_cols:
    full[c] = full[c].astype("category")

train_feat = full.iloc[:len(train_df)].copy()
test_feat  = full.iloc[len(train_df):].copy()

# 3. STAGE 1 CLASSIFICATION
y_stage1 = (train_df[TARGET] != "Perfect_Trip").astype(int)

X_stage1 = train_feat.drop(
    columns=[ID_COL, "is_normal_dist", "is_normal_price"]
)

X_tr, X_va, y_tr, y_va = train_test_split(
    X_stage1,
    y_stage1,
    test_size=0.15,
    stratify=y_stage1,
    random_state=42
)

stage1 = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=600,          
    learning_rate=0.06,
    num_leaves=48,             
    min_data_in_leaf=120,      
    subsample=0.85,
    colsample_bytree=0.85,
    class_weight={0:1.0, 1:1.7},  
    random_state=42
)

stage1.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(60)]
)

# Threshold MICRO tuning
proba_va = stage1.predict_proba(X_va)[:,1]

best_t, best_p = 0.38, 0
for t in np.linspace(0.36, 0.42, 25):
    pred = (proba_va >= t).astype(int)
    r = recall_score(y_va, pred)
    p = precision_score(y_va, pred)
    if r >= 0.72 and p > best_p:
        best_t, best_p = t, p

print(f"\nSTAGE-1 FINAL THRESHOLD = {best_t:.3f}")
print(classification_report(y_va, (proba_va >= best_t).astype(int)))

# SAVE STAGE-1 ARTIFACTS 
joblib.dump(stage1, "stage1_lgbm_model.joblib")
joblib.dump(best_t, "stage1_best_threshold.joblib")
print("✅ Stage-1 model & threshold saved")


# 4. STAGE 2 — MULTICLASS CLASSIFICATION
mask = train_df[TARGET] != "Perfect_Trip"
X_stage2 = train_feat.loc[mask].drop(columns=[ID_COL])
y_stage2 = train_df.loc[mask, TARGET]

vc = y_stage2.value_counts()
valid_classes = vc[vc >= 2].index
mask_valid = y_stage2.isin(valid_classes)

X_stage2 = X_stage2.loc[mask_valid]
y_stage2 = y_stage2.loc[mask_valid]

le = LabelEncoder()
y_enc = le.fit_transform(y_stage2)

X_tr2, X_va2, y_tr2, y_va2 = train_test_split(
    X_stage2, y_enc,
    test_size=0.15,
    stratify=y_enc,
    random_state=42
)

stage2 = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    n_estimators=1200,
    learning_rate=0.04,
    num_leaves=72,
    min_data_in_leaf=40,
    subsample=0.85,
    colsample_bytree=0.85,
    class_weight="balanced",
    random_state=42
)

stage2.fit(
    X_tr2, y_tr2,
    eval_set=[(X_va2, y_va2)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(100)]
)

# SAVE STAGE-2 ARTIFACTS 
joblib.dump(stage2, "stage2_lgbm_model.joblib")
joblib.dump(le, "stage2_label_encoder.joblib")
print("✅ Stage-2 model & label encoder saved")

# 5. FINAL INFERENCE
X_test_stage1 = test_feat.drop(
    columns=[ID_COL, "is_normal_dist", "is_normal_price"]
)

p_test = stage1.predict_proba(X_test_stage1)[:,1]
stage1_test = (p_test >= best_t).astype(int)

final_pred = np.array(["Perfect_Trip"] * len(test_feat), dtype=object)
problem_idx = np.where(stage1_test == 1)[0]

if len(problem_idx) > 0:
    probs2 = stage2.predict_proba(
        test_feat.iloc[problem_idx].drop(columns=[ID_COL])
    )
    final_pred[problem_idx] = le.inverse_transform(probs2.argmax(axis=1))

submission = pd.DataFrame({
    "Trip_ID": test_feat[ID_COL].values,
    "Trip_Label": final_pred
})

submission.to_csv("submission_final_tuned.csv", index=False)
submission.head()

# SAVE FULL PIPELINE METADATA 
pipeline_meta = {
    "cat_cols": cat_cols,
    "id_col": ID_COL,
    "target": TARGET
}

joblib.dump(pipeline_meta, "pipeline_meta.joblib")
print("✅ Pipeline metadata saved")


# score: 0.59376

[LightGBM] [Warning] min_data_in_leaf is set=120, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=120
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=120, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=120
[LightGBM] [Info] Number of positive: 3062034, number of negative: 3737966
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.387726 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43286
[LightGBM] [Info] Number of data points in the train set: 6800000, number of used features: 30
[LightGBM] [Warning] min_data_in_leaf is set=120, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=120
[LightGBM] [Info] [binary:Boos

# Second Best Model

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score

pd.set_option("display.max_columns", None)

# 1. LOAD DATA
train_df = pd.read_csv("/kaggle/input/anava-ugm/train.csv")
test_df  = pd.read_csv("/kaggle/input/anava-ugm/test.csv")

TARGET = "Trip_Label"
ID_COL = "Trip_ID"

# 2. FEATURE ENGINEERING
full = pd.concat(
    [train_df.drop(columns=[TARGET]), test_df],
    axis=0
).reset_index(drop=True)

full["Timestamp"] = pd.to_datetime(full["Timestamp"], errors="coerce")
full["Timestamp"] = full["Timestamp"].fillna(full["Timestamp"].median())

for c in ["Pickup_Lat","Pickup_Long","Dropoff_Lat","Dropoff_Long"]:
    full[c] = full[c].fillna(full[c].median())

full["Battery_Level"] = (
    full["Battery_Level"].astype(str)
    .str.replace("%","", regex=False)
)
full["Battery_Level"] = pd.to_numeric(full["Battery_Level"], errors="coerce")
full["Battery_Level"] = full["Battery_Level"].fillna(full["Battery_Level"].median())

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2-lon1, lat2-lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

full["haversine_km"] = haversine(
    full["Pickup_Long"], full["Pickup_Lat"],
    full["Dropoff_Long"], full["Dropoff_Lat"]
)

full["dist_ratio"] = full["haversine_km"] / (full["Distance_KM"] + 1e-3)

full["Surge_Multiplier"] = full["Surge_Multiplier"].fillna(1.0)
full["price_per_km"] = full["Est_Price_IDR"] / (full["Distance_KM"] + 1e-3)
full["surge_flag"] = (full["Surge_Multiplier"] > 1).astype(int)

zone_avg = full.groupby("Pickup_Zone")["price_per_km"].transform("mean")
full["price_vs_zone"] = full["price_per_km"] / (zone_avg + 1e-3)

full["promo_flag"] = full["Promo_Code"].notna().astype(int)
full.drop(columns=["Promo_Code"], inplace=True)

full["Hour"] = full["Timestamp"].dt.hour
full["DayOfWeek"] = full["Timestamp"].dt.dayofweek
full["IsWeekend"] = (full["DayOfWeek"] >= 5).astype(int)
full.drop(columns=["Timestamp"], inplace=True)

full["is_normal_dist"]  = (np.abs(full["dist_ratio"] - 1) < 0.15).astype(int)
full["is_normal_price"] = (np.abs(full["price_vs_zone"] - 1) < 0.15).astype(int)

cat_cols = [
    "Pickup_Zone","Dropoff_Zone","Device_FP","Car_Model",
    "Payment_Method","Weather","Traffic","Signal_Strength"
]
for c in cat_cols:
    full[c] = full[c].astype("category")

train_feat = full.iloc[:len(train_df)].copy()
test_feat  = full.iloc[len(train_df):].copy()

# 3. STAGE 1 — BINARY CLASSIFICATION
y_stage1 = (train_df[TARGET] != "Perfect_Trip").astype(int)

X_stage1 = train_feat.drop(
    columns=[ID_COL, "is_normal_dist", "is_normal_price"]
)

X_tr, X_va, y_tr, y_va = train_test_split(
    X_stage1,
    y_stage1,
    test_size=0.15,
    stratify=y_stage1,
    random_state=42
)

stage1 = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=900,
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=80,
    subsample=0.85,
    colsample_bytree=0.85,
    class_weight={0:1.0, 1:1.6},
    random_state=42
)

stage1.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(80)]
)

# Threshold tuning
proba_va = stage1.predict_proba(X_va)[:,1]

best_t, best_p = 0.5, 0
for t in np.linspace(0.10, 0.45, 70):
    pred = (proba_va >= t).astype(int)
    r = recall_score(y_va, pred)
    p = precision_score(y_va, pred)
    if r >= 0.72 and p > best_p:
        best_t, best_p = t, p

print(f"\nSTAGE 1 THRESHOLD = {best_t:.3f}")
print(classification_report(y_va, (proba_va >= best_t).astype(int)))

# SAVE STAGE 1
joblib.dump(stage1, "stage1_model.joblib")
joblib.dump(best_t, "stage1_threshold.joblib")
joblib.dump(X_stage1.columns.tolist(), "stage1_features.joblib")

# 4. STAGE 2 — MULTICLASS CLASSIFICATION
mask = train_df[TARGET] != "Perfect_Trip"
X_stage2 = train_feat.loc[mask].drop(columns=[ID_COL])
y_stage2 = train_df.loc[mask, TARGET]

vc = y_stage2.value_counts()
valid_classes = vc[vc >= 2].index
mask_valid = y_stage2.isin(valid_classes)

X_stage2 = X_stage2.loc[mask_valid]
y_stage2 = y_stage2.loc[mask_valid]

le = LabelEncoder()
y_enc = le.fit_transform(y_stage2)

X_tr2, X_va2, y_tr2, y_va2 = train_test_split(
    X_stage2, y_enc,
    test_size=0.15,
    stratify=y_enc,
    random_state=42
)

stage2 = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    n_estimators=1200,
    learning_rate=0.04,
    num_leaves=72,
    min_data_in_leaf=40,
    subsample=0.85,
    colsample_bytree=0.85,
    class_weight="balanced",
    random_state=42
)

stage2.fit(
    X_tr2, y_tr2,
    eval_set=[(X_va2, y_va2)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(100)]
)

print("\nSTAGE 2 REPORT\n")
print(classification_report(y_va2, stage2.predict(X_va2), target_names=le.classes_))

# SAVE STAGE 2
joblib.dump(stage2, "stage2_model.joblib")
joblib.dump(le, "label_encoder.joblib")
joblib.dump(X_stage2.columns.tolist(), "stage2_features.joblib")

# 5. FINAL INFERENCE
X_test_stage1 = test_feat[X_stage1.columns]

p_test = stage1.predict_proba(X_test_stage1)[:,1]
stage1_test = (p_test >= best_t).astype(int)

final_pred = np.array(["Perfect_Trip"] * len(test_feat), dtype=object)
problem_idx = np.where(stage1_test == 1)[0]

if len(problem_idx) > 0:
    probs2 = stage2.predict_proba(
        test_feat.iloc[problem_idx][X_stage2.columns]
    )
    final_pred[problem_idx] = le.inverse_transform(probs2.argmax(axis=1))

submission = pd.DataFrame({
    "Trip_ID": test_feat[ID_COL].values,
    "Trip_Label": final_pred
})

submission.to_csv("submission_final_2stage_safe.csv", index=False)
submission.head()

# score: 0.59363

[LightGBM] [Warning] min_data_in_leaf is set=80, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=80
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=80, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=80
[LightGBM] [Info] Number of positive: 3062034, number of negative: 3737966
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.194655 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43286
[LightGBM] [Info] Number of data points in the train set: 6800000, number of used features: 30
[LightGBM] [Warning] min_data_in_leaf is set=80, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=80
[LightGBM] [Info] [binary:BoostFromS

,Trip_ID,Trip_Label
0,TRIP-06583736,Navigation_Issue
1,TRIP-11356251,Perfect_Trip
2,TRIP-03320505,Perfect_Trip
3,TRIP-07188814,Navigation_Issue
4,TRIP-06994869,Navigation_Issue
